# [실습] 피처 선택 · 데이터 누수

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

# 🔧 [실습] 피처·파이프라인·데이터 누수 — 새지 않는 실험 설계

## — 사용하지 않았던 범주형 피처를 평가 정보가 새지 않게 연결합니다

지난 순서에 학습곡선이 이렇게 말했습니다. _"데이터를 더 모아도 소용없다. 입력을 바꿔라."_ 그런데 우리는 원본의 **17개 예측 피처 중 수치형 10개만** 사용하고 있었습니다. 문자 열을 모델에 넣는 법을 몰랐기 때문입니다.

오늘 범주형 7개를 추가해 17개 원본 예측 피처를 모두 검토합니다. 그리고 그 과정에서 평가를 왜곡할 수 있는 사고를 다루는 법을 함께 배웁니다.

> 🤖 **오늘의 AI 활용 규칙 — 검증 단계:**  
> 전처리 코드는 AI에게 물어도 됩니다. 단, 받은 코드에서 **`fit`의 대상, 피처 생성 시점, 분할 단위**를 확인합니다. 이 한 줄 점검이 오늘 배우는 것의 가장 실용적인 쓰임입니다. (자세한 규칙은 개념 노트북 Part 0)

## 📋 오늘의 실습

지난 보고를 받은 팀장이 승인했습니다.

> 🧑‍💼 "버리고 있던 열을 쓰는 건 좋습니다. 그런데 옆 팀이 비슷한 걸 했다가 **실서비스에서 성능이 반토막** 난 적이 있어요. 개발할 때 0.95였는데 배포하니 0.7이었다고 합니다. 그런 일이 없다는 걸 어떻게 보장하시겠습니까?"

팀장이 말한 사고가 바로 **데이터 누수**입니다. 오늘은 성능을 올리는 일과, **그 성능 추정이 타당한지 확인하는 일**을 함께 합니다.

| 문제 | 내용                                   | 확인하는 힘                     |
| ---- | -------------------------------------- | ------------------------------- |
| 1    | 범주형 7개 열을 `Pipeline`으로 푼다    | `ColumnTransformer` 구성        |
| 2    | 파생변수 가설 2개를 세우고 검증한다    | 가설 → 실험 → 채택/기각         |
| 3    | 누수를 직접 만들어 부풀림을 잰다       | 누수 진단과 교차 적합           |
| 4    | `Pipeline` 통째로 `GridSearchCV`       | 전처리까지 함께 튜닝            |
| 5    | 실험 기록표 · 모델 저장 · 모델 카드 v5 | 재현 가능한 산출물 (**제출물**) |

> 📌 **오늘의 주 지표는 AP와 F1입니다.**  
> `average_precision`으로 계산한 AP는 여러 임계값의 정밀도·재현율을 요약하고, F1은 기본 임계값의 운영점을 보여줍니다. 정확도는 보조 지표로 확인합니다.

# ⚙️ 데이터 준비

이론 노트북의 Titanic과 달리, 이번 실습 노트북에서는 이전 시간에서 사용한 **UCI Online Shoppers Purchasing Intention** 데이터를 이어서 사용합니다. 한 행은 온라인 쇼핑몰 방문 세션 1건이며, 세션 정보로 구매 완료 여부를 분류합니다.

| 항목        | 내용                                                                      |
| ----------- | ------------------------------------------------------------------------- |
| 관측 단위   | 온라인 쇼핑몰 방문 세션 1건 — 1년 동안 세션별 사용자가 겹치지 않도록 구성 |
| 데이터 크기 | 12,330행 × 18열 — 입력 피처 17개 + 타깃 1개                               |
| 타깃        | `Revenue` — 구매 완료 `True`(1), 미구매 `False`(0)                        |
| 클래스 분포 | 구매 1,908건(15.5%) · 미구매 10,422건(84.5%)                              |
| 입력 피처   | 수치형 10개 + 범주형 7개                                                  |
| 결측치      | 없음                                                                      |
| 사용 목적   | 혼합형 전처리, 파생변수 비교, 타깃 인코딩 누수 재현과 교차 적합           |

### 입력 피처 구성

- **수치형 10개:** `Administrative`부터 `SpecialDay`까지의 방문 행동·시간·페이지 지표
- **범주형 7개:** `Month`, `OperatingSystems`, `Browser`, `Region`, `TrafficType`, `VisitorType`, `Weekend`
- **타깃 1개:** `Revenue`

`OperatingSystems`·`Browser`·`Region`·`TrafficType`은 숫자로 저장돼 있지만 **크기에 의미가 없는 코드**입니다(`Browser=13`이 `Browser=1`보다 크다는 뜻이 아닙니다). 따라서 이 네 열은 범주형 전처리 갈래에 배정합니다.

> ℹ️ **출처와 이용 조건**  
> [UCI Machine Learning Repository · Dataset 468](https://archive.ics.uci.edu/dataset/468/online%2Bshoppers%2Bpurchasing%2Bintention%2Bdataset), DOI `10.24432/C5F88Q` · **CC BY 4.0**

> ⚠️ **예측 시점 먼저 정하기**  
> 이 실습은 세션 정보로 같은 세션의 구매 완료 여부를 설명합니다. 실시간 구매 의도 예측으로 확장하려면, 예측 요청 시점까지 확정된 피처만 남겨야 합니다. 특히 `PageValues`는 거래 완료와 관련해 계산되므로 시점 누수 후보로 따로 점검합니다.

▶️ **코드 실행하기 · 코드 셀 1 [C1]**

In [1]:
# ─────────────────────────────────────────────
# [C1] ⚙️ 데이터 준비
# 최초 1회 다운로드 → data/ 폴더에 저장 (이후 오프라인)
# ─────────────────────────────────────────────
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def fetch_uci(url, zip_name, member):
    """UCI 정적 저장소의 zip을 내려받아 data/에 풀고, CSV 경로를 돌려줍니다."""
    csv_path = DATA_DIR / member
    if not csv_path.exists():
        zip_path = DATA_DIR / zip_name
        if not zip_path.exists():
            print(f"내려받는 중… {zip_name}")
            urllib.request.urlretrieve(url, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            z.extract(member, DATA_DIR)
    return csv_path

shoppers = pd.read_csv(fetch_uci(
    "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip",
    "online_shoppers.zip", "online_shoppers_intention.csv"))

NUM_COLS = ["Administrative", "Administrative_Duration", "Informational",
            "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
            "BounceRates", "ExitRates", "PageValues", "SpecialDay"]
CAT_COLS = ["Month", "OperatingSystems", "Browser", "Region",
            "TrafficType", "VisitorType", "Weekend"]

y = shoppers["Revenue"].astype(int)

print(f"수치형 {len(NUM_COLS)}개 · 범주형 {len(CAT_COLS)}개 · 타깃 1개")
print("범주별 값 개수:", {c: shoppers[c].nunique() for c in CAT_COLS})
print("\n→ 준비 완료. 이제 여러분 차례입니다.")

내려받는 중… online_shoppers.zip
수치형 10개 · 범주형 7개 · 타깃 1개
범주별 값 개수: {'Month': 10, 'OperatingSystems': 8, 'Browser': 13, 'Region': 9, 'TrafficType': 20, 'VisitorType': 3, 'Weekend': 2}

→ 준비 완료. 이제 여러분 차례입니다.


# 문제 1. 범주형 7개 열을 `Pipeline`으로 푼다

`ColumnTransformer`로 수치형은 그대로 통과시키고 범주형만 One-Hot으로 펼칩니다. 그 전체를 `Pipeline`에 담아 **교차 검증에 통째로** 넘깁니다. 그러면 인코더가 겹마다 학습 데이터로만 `fit`됩니다.

```
[문제 1]
1) ColumnTransformer를 만듭니다.
   - 수치형: "passthrough"
   - 범주형: OneHotEncoder(handle_unknown="ignore", min_frequency=20)
2) 그것과 지난 순서의 모델을 Pipeline으로 묶습니다.
   RandomForestClassifier(n_estimators=300, min_samples_leaf=20)
3) 5겹 CV로 F1과 AP(`average_precision`)를 측정하고, 수치형 10개만 썼을 때와 비교합니다.
4) 변환 후 피처가 몇 개가 됐는지 출력합니다.
```

> 🤔 **예상하기**  
> 열이 10개에서 17개로 늘고, One-Hot으로 펼치면 피처는 더 많아집니다. F1과 AP의 변화 방향을 예상합니다. 두 지표가 같은 방향으로 움직일지 확인합니다.

▶️ **코드 실행하기 · 코드 셀 2 [C2]**

In [2]:
# [C2] 문제 1. 범주형 7개 열을 `Pipeline`으로 푼다
# ⌨️ 문제 1 — ColumnTransformer + Pipeline으로 범주형 개방
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier

# 여기에 코드를 작성하세요 (ColumnTransformer → Pipeline → CV 비교 → 피처 수)
# 1) 전처리기
pre = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
])

# 2) 파이프라인
pipe = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                    random_state=RANDOM_STATE, n_jobs=-1)),
])

# 3) 5겹 CV — 전체 피처 vs 수치형 10개만
scoring = ["f1", "average_precision"]

res_full = cross_validate(pipe, shoppers[NUM_COLS + CAT_COLS], y, cv=5, scoring=scoring)
res_num = cross_validate(
    RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                            random_state=RANDOM_STATE, n_jobs=-1),
    shoppers[NUM_COLS], y, cv=5, scoring=scoring
)

print(f"전체 피처   F1: {res_full['test_f1'].mean():.4f}  AP: {res_full['test_average_precision'].mean():.4f}")
print(f"수치형만    F1: {res_num['test_f1'].mean():.4f}  AP: {res_num['test_average_precision'].mean():.4f}")

# 4) 변환 후 피처 개수
n_features = pre.fit_transform(shoppers[NUM_COLS + CAT_COLS]).shape[1]
print(f"변환 후 피처 수: {n_features}")

전체 피처   F1: 0.5447  AP: 0.7162
수치형만    F1: 0.6120  AP: 0.7171
변환 후 피처 수: 66


<details>
<summary>(클릭) 💡 힌트</summary>

- `ColumnTransformer([("num", "passthrough", NUM_COLS), ("cat", OneHotEncoder(...), CAT_COLS)])`
- `min_frequency=20`은 20건 미만으로 나타나는 희귀 값을 하나로 묶습니다. 열이 지나치게 늘어나는 것을 막습니다.
- `handle_unknown="ignore"`가 없으면 검증 겹에만 나타나는 값에서 오류가 납니다.
- `Pipeline([("pre", 전처리), ("clf", 모델)])`을 통째로 `cross_validate`에 넘깁니다.
- 변환 후 피처 수는 전처리를 `fit`한 뒤 `.transform(X).shape[1]`로 확인합니다.
- 비교 대상(수치형만)은 `Pipeline` 없이 `cross_validate(model, shoppers[NUM_COLS], y, ...)`로 잽니다.

</details>

> 🎯 **[C2] 확인하기**  
> **한쪽만 올랐습니다.** AP는 0.7248 → **0.7372**로 올랐는데, F1은 0.6204 → **0.5780**으로 오히려 떨어졌습니다.
>
> 모순처럼 보이지만 아닙니다. 지난 순서에서 배운 내용을 연결합니다 — **AP는 여러 임계값의 정밀도·재현율을 요약**하고, **F1은 기본 판정 임계값에서** 계산됩니다. 범주형을 넣어 모델의 *확률 순위 매기는 능력*은 좋아졌는데(AP↑), 확률 분포가 바뀌면서 **0.5라는 선이 전보다 나쁜 자리**에 놓인 것입니다.
>
> 실무에서 이럴 때 하는 일은 명확합니다. F1 하락만으로 범주형을 즉시 제외하지 않습니다. 먼저 검증 데이터에서 임계값을 다시 선택하고 AP와 운영 제약을 함께 확인합니다. 지표가 무엇을 재는지 알면 이런 판단이 가능해집니다.

# 문제 2. 파생변수 — 가설을 먼저 적고, 그다음 검증한다

피처 엔지니어링의 순서는 **코드가 먼저가 아닙니다.** 가설이 먼저입니다. 아무 조합이나 만들어 점수가 오르길 기다리는 것은 실험이 아니라 도박입니다.

다행히 우리에겐 근거가 있습니다. **순서 2의 실습에서 이미 발견한 사실**이 있습니다.

- 이 데이터에서 `PageValues`가 0인 세션의 구매율 **3.9%**, 0보다 큰 세션은 **56.3%** — 무려 14.6배
- 구매 세션은 상품 페이지 체류가 길다 (중앙값 위 22.3% 대 아래 8.7%)

```
[문제 2]
1) 아래 두 가설을 코드로 만들기 전에, 각각 "왜 효과가 있을 것인가"를 한 줄로 적습니다.
   가설 A: has_page_value = (PageValues > 0)  ← 0 여부가 추가 표현으로 유용할 수 있다
   가설 B: total_pages = Administrative + Informational + ProductRelated  ← 총 탐색량
2) 문제 1의 ② 구성을 기준으로, A만 / B만 각각 추가해 5겹 CV로 측정합니다.
3) 각각의 ΔF1과 ΔAP를 계산해 채택·기각을 결정합니다.
```

> 🤔 **예상하기**  
> `has_page_value`는 이미 있는 `PageValues`에서 연속값을 0/1로 단순화한 피처입니다. 트리가 자체적으로 분기점을 찾을 수 있는데도 이 표현이 추가 효과를 낼지 예상합니다.

> ⚠️ **주의하기 — 예측 시점**  
> UCI 설명에서 `PageValues`는 전자상거래 거래 완료와 관련해 계산되는 지표입니다. 실제 서비스에 사용하려면 예측 요청 시점에 이 값이 이미 확정되어 있는지 확인해야 합니다. 확인되지 않으면 시점 누수 후보로 분류합니다.

▶️ **코드 실행하기 · 코드 셀 3 [C3]**

In [3]:
# [C3] 문제 2. 파생변수 — 가설을 먼저 적고, 그다음 검증한다
# ⌨️ 문제 2 — 가설 A·B를 각각 따로 검증

# 여기에 코드를 작성하세요 (파생 열 추가 → A만 / B만 → Δ 계산 → 채택 결정)
# 1)
# 가설 A: has_page_value = (PageValues > 0)
# → PageValues가 0인 세션은 구매율 3.9%, 0보다 큰 세션은 56.3%로 차이가 극단적이니, "0이냐 아니냐"라는 이분법 자체가 강한 신호일 것이다.

# 가설 B: total_pages = Administrative + Informational + ProductRelated
# → 구매 세션은 페이지 탐색이 활발했으니(체류시간 22.3% vs 8.7%), 총 방문 페이지 수도 구매 여부와 관련 있을 것이다.

# 2)
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

def eval_numeric_features(cols, X_df, y, random_state=RANDOM_STATE):
    """수치형 컬럼 목록만 받아서 5겹 CV로 F1, AP 평균을 돌려주는 함수"""
    model = RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                    random_state=random_state, n_jobs=-1)
    res = cross_validate(model, X_df[cols], y, cv=5,
                          scoring=["f1", "average_precision"])
    return res["test_f1"].mean(), res["test_average_precision"].mean()

# 원본 건드리지 않게 복사
feat = shoppers.copy()
feat["has_page_value"] = (feat["PageValues"] > 0).astype(int)
feat["total_pages"] = feat["Administrative"] + feat["Informational"] + feat["ProductRelated"]

# 기준선: 문제1의 수치형 10개만 (이미 res_num으로 구해놓은 값)
base_f1, base_ap = res_num["test_f1"].mean(), res_num["test_average_precision"].mean()

# 가설 A만 추가
cols_A = NUM_COLS + ["has_page_value"]
f1_A, ap_A = eval_numeric_features(cols_A, feat, y)

# 가설 B만 추가
cols_B = NUM_COLS + ["total_pages"]
f1_B, ap_B = eval_numeric_features(cols_B, feat, y)

print(f"기준(수치형만)   F1={base_f1:.4f}  AP={base_ap:.4f}")
print(f"A 추가(has_page) F1={f1_A:.4f} (Δ{f1_A-base_f1:+.4f})  AP={ap_A:.4f} (Δ{ap_A-base_ap:+.4f})")
print(f"B 추가(total_pg) F1={f1_B:.4f} (Δ{f1_B-base_f1:+.4f})  AP={ap_B:.4f} (Δ{ap_B-base_ap:+.4f})")

기준(수치형만)   F1=0.6120  AP=0.7171
A 추가(has_page) F1=0.6139 (Δ+0.0019)  AP=0.7183 (Δ+0.0012)
B 추가(total_pg) F1=0.6077 (Δ-0.0043)  AP=0.7157 (Δ-0.0013)


In [ ]:
# 3)
# 가설 A (has_page_value): 기각
# ΔF1 +0.0019, ΔAP +0.0012로 노이즈 수준. RandomForest가 이미 PageValues의 분기점을 스스로 찾아내므로 추가 표현이 불필요.

# 가설 B (total_pages): 기각
# ΔF1 -0.0043, ΔAP -0.0013로 오히려 하락. 기존 컬럼들의 단순 합이라 새 정보가 없고, 원본 컬럼과 공존하며 트리 분기만 방해.

<details>
<summary>(클릭) 💡 힌트</summary>

- 원본을 건드리지 않도록 `feat = shoppers.copy()`로 시작합니다.
- 불리언을 피처로 넣을 때는 `.astype(int)`로 0/1을 만듭니다.
- 두 가설을 **따로** 재야 어느 쪽이 효과를 냈는지 알 수 있습니다. 한꺼번에 넣으면 원인을 못 가립니다.
- `ColumnTransformer`의 수치 목록만 바꾸면 되므로, 목록을 인자로 받는 함수로 만들어 두면 반복이 줄어듭니다.
- 기준은 문제 1의 ② 결과입니다 — `log[1]`에 들어 있습니다.

</details>

> 🎯 **[C3] 확인하기**  
> **0 여부를 표현한 피처의 평균 점수가 높아졌습니다.** `has_page_value`는 연속값을 0/1로 줄이기만 했는데 F1을 **+0.0687** 올렸습니다 — 오늘 얻은 개선 중 가장 큽니다.
>
> 랜덤포레스트는 `PageValues`의 분기점을 자체적으로 찾을 수 있지만, 0 여부를 명시한 표현이 제한된 트리 구조에서 더 쉽게 선택됐을 가능성이 있습니다. 정확한 원인은 추가 실험 없이는 단정할 수 없습니다.
>
> 이 가설의 출처도 확인합니다. **순서 2의 실습에서 직접 계산한 구매율 대비**입니다. 파생변수는 관찰에서 가설을 세우고, 별도 검증과 예측 시점 감사로 채택 여부를 결정합니다.

# 문제 3. 누수를 직접 만들어, 부풀림을 잰다

이제 팀장의 질문에 답할 차례입니다. 개발 점수가 낙관적으로 나타나는 과정을 직접 재현합니다.

흔한 시나리오 하나를 씁니다. 범주 조합별 **과거 구매율**을 피처로 만드는 것입니다(타깃 인코딩). 아이디어 자체는 정상이지만, **전체 데이터로 계산해 버리면** 검증 겹의 정답이 피처에 스며듭니다.

```
[문제 3]
1) 조합 키를 만듭니다 — combo = Month + "_" + TrafficType + "_" + Region
   (몇 개 조합이 나오는지, 10건 미만인 조합이 몇 개인지 확인합니다)
2) 누수 버전: 전체 데이터로 조합별 Revenue 평균을 구해 열로 추가 → CV
3) 교차 적합 버전: 같은 일을 `TargetEncoder`로 `Pipeline` 안에서 수행 → CV
4) 두 점수의 차이를 계산하고, 교차 적합 버전이 기준(문제 1의 ②)보다 나아졌는지 확인합니다.
```

> 🤔 **예상하기**  
> 조합 키는 715개가 나오고 그중 464개는 10건 미만입니다. 표본이 적은 그룹의 구매율에 각 행의 정답이 얼마나 크게 반영될지 예상합니다. 전체 데이터로 계산했을 때 검증 행의 정답이 피처에 들어가는 경로를 확인합니다.

▶️ **코드 실행하기 · 코드 셀 4 [C4]**

In [5]:
# [C4] 문제 3. 누수를 직접 만들어, 부풀림을 잰다
# ⌨️ 문제 3 — 같은 피처를 누수 버전과 정상 버전으로 각각 재기
from sklearn.preprocessing import TargetEncoder

# 여기에 코드를 작성하세요 (조합 키 → 누수 버전 CV → Pipeline 안 버전 CV → 차이)
# 1) 조합 키 확인
combo = shoppers["Month"].astype(str) + "_" + shoppers["TrafficType"].astype(str) + "_" + shoppers["Region"].astype(str)

print("조합 개수:", combo.nunique())
counts = combo.value_counts()
print("10건 미만 조합 수:", (counts < 10).sum())

# 2) 누수 버전 CV
feat_leak = shoppers.copy()
feat_leak["combo"] = combo
feat_leak["combo_target_mean"] = feat_leak.groupby("combo")["Revenue"].transform("mean")

cols_leak = NUM_COLS + ["combo_target_mean"]
f1_leak, ap_leak = eval_numeric_features(cols_leak, feat_leak, y)
print(f"누수 버전  F1={f1_leak:.4f}  AP={ap_leak:.4f}")

# 3) Pipeline 안 버전 CV
feat_cf = shoppers.copy()
feat_cf["combo"] = combo

pre_te = ColumnTransformer([
    ("num", "passthrough", NUM_COLS),
    ("te", TargetEncoder(random_state=RANDOM_STATE), ["combo"]),
])

pipe_te = Pipeline([
    ("pre", pre_te),
    ("clf", RandomForestClassifier(n_estimators=300, min_samples_leaf=20,
                                    random_state=RANDOM_STATE, n_jobs=-1)),
])

res_te = cross_validate(pipe_te, feat_cf[NUM_COLS + ["combo"]], y, cv=5,
                         scoring=["f1", "average_precision"])
f1_te, ap_te = res_te["test_f1"].mean(), res_te["test_average_precision"].mean()
print(f"교차적합 버전  F1={f1_te:.4f}  AP={ap_te:.4f}")

# 4) 비교
print(f"기준(수치형만)     F1={base_f1:.4f}  AP={base_ap:.4f}")
print(f"누수 버전         F1={f1_leak:.4f}  AP={ap_leak:.4f}  (Δ{f1_leak-base_f1:+.4f})")
print(f"교차적합 버전      F1={f1_te:.4f}  AP={ap_te:.4f}  (Δ{f1_te-base_f1:+.4f})")

조합 개수: 715
10건 미만 조합 수: 464
누수 버전  F1=0.6572  AP=0.7543


c:\Users\ahnny\anaconda3\envs\myenv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\ahnny\anaconda3\envs\myenv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\ahnny\anaconda3\envs\myenv\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:341: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to speci

교차적합 버전  F1=0.6193  AP=0.7182
기준(수치형만)     F1=0.6120  AP=0.7171
누수 버전         F1=0.6572  AP=0.7543  (Δ+0.0451)
교차적합 버전      F1=0.6193  AP=0.7182  (Δ+0.0073)


In [6]:
# 누수 버전: F1=0.6572(Δ+0.0451), AP=0.7543(Δ+0.0372) - 전체 데이터로 조합별 구매율을 계산해 검증 폴드의 정답이 피처에 섞임. 10건 미만 조합이 464개(65%)라 소수 표본의 평균이 사실상 정답을 알려줌 → 점수가 부풀려짐.
# 교차적합 버전: F1=0.6197(Δ+0.0077), AP=0.7182(Δ+0.0011) - TargetEncoder를 Pipeline 안에 넣어 폴드마다 학습 데이터로만 fit. 실제 개선 효과는 거의 없음.
# 결론: 누수 버전의 개선폭 대부분(4~5배)은 실력이 아니라 데이터 누수로 인한 착시. 조합 타깃 인코딩 피처는 기각.

<details>
<summary>(클릭) 💡 힌트</summary>

- 조합 키는 문자열을 이어 붙여 만듭니다: `df["Month"].astype(str) + "_" + df["TrafficType"].astype(str) + ...`
- **누수 버전**의 핵심은 `groupby(...)["Revenue"].transform("mean")` — 이 한 줄이 전체 데이터를 보고 계산합니다.
- **정상 버전**은 `TargetEncoder`를 `ColumnTransformer`의 한 갈래로 넣으면 됩니다. 그러면 겹마다 학습 데이터로만 `fit`됩니다.
- 두 버전 모두 **나머지 조건은 똑같이** 두어야 비교가 성립합니다.
- 조합 개수는 `.nunique()`, 10건 미만 그룹 수는 `(값별_개수 < 10).sum()`입니다.

</details>

> 🎯 **[C4] 확인하기**  
> **10건짜리 그룹의 "과거 구매율"은 사실상 그 10건의 정답을 평균한 값**입니다. 자기 자신의 답을 포함한 채로요. 715개 조합 중 464개가 그런 상태입니다.
>
> 그래서 부풀림이 **AP +0.0125, F1 +0.0264** 만큼 생겼습니다. 더 무서운 것은 방향입니다 — 누수 버전만 보면 이 피처가 **기준보다 +0.011 좋아 보이는데**, 정직하게 재면 **−0.002로 오히려 나쁩니다.** 개선이 있다고 믿고 배포했다면, 운영 데이터에서는 미래 타깃으로 같은 값을 계산할 수 없으므로 개발 점수가 재현되지 않을 수 있습니다.
>
> 이 실험은 타깃 인코딩의 계산 범위가 평가를 어떻게 왜곡하는지 보여줍니다. `TargetEncoder`를 `Pipeline` 안에 두면 학습 시 내부 교차 적합이 적용되지만, 예측 시점 이후 피처와 분할 단위는 별도로 감사해야 합니다.

> 📌 **오늘의 점검 한 줄:**  
> 남의 코드(또는 AI가 준 코드)를 볼 때 가장 먼저 찾을 것은 **학습되는 `fit`·`fit_transform`이나 타깃을 사용하는 `groupby(...).transform`이 교차 검증 바깥에서 실행되는가**입니다. 바깥에 있다면 그 숫자는 일단 의심해야 합니다.

# 문제 4. `Pipeline` 통째로 `GridSearchCV`

지금까지 손잡이를 하나씩 손으로 돌렸습니다. 이제 자동화합니다. 그런데 핵심은 **전처리 옵션도 함께 튜닝한다**는 점입니다 — `Pipeline`을 쓰면 전처리와 모델의 손잡이를 **한 격자에서** 다룰 수 있습니다.

> 🔁 **개념 복습 — 이번 탐색의 규모**  
> 전처리 옵션 2개 × 모델 옵션 3개로 **6개 후보 조합**을 만듭니다. 각 조합을 같은 5겹으로 평가하므로 후보 비교에 6 × 5 = **30번의 학습**이 필요합니다. `average_precision`의 평균이 가장 높은 조합을 선택하고, 기본값 `refit=True`가 선택된 `Pipeline`을 전체 입력 데이터로 한 번 더 학습합니다. `best_score_`는 이 선택 과정의 내부 CV 점수입니다.

```
[문제 4]
1) 문제 2에서 채택한 구성(수치 10개 + has_page_value + 범주 7개)으로 Pipeline을 만듭니다.
2) GridSearchCV로 다음 두 손잡이를 함께 탐색합니다. `scoring="average_precision"`
   - pre__cat__min_frequency: [10, 50]      ← 전처리 옵션
   - clf__min_samples_leaf:   [5, 20, 50]   ← 모델 옵션
3) 조합별 내부 CV 점수를 표로 출력하고, `best_params_`와 `best_score_`를 확인합니다.
4) 선택된 모델의 F1도 같은 5겹 분할에서 측정해 기록합니다. 이 값은 모델 선택에 사용한 데이터의 내부 추정치이므로 최종 일반화 성능으로 보고하지 않습니다.
```

> 🤔 **예상하기**  
> 지난 순서에서 `min_samples_leaf`는 **20**이 좋았습니다. 그런데 그때는 피처가 10개뿐이었습니다. 피처 표현이 늘어난 뒤에도 20이 선택될지 예상합니다.

▶️ **코드 실행하기 · 코드 셀 5 [C5]**

In [7]:
# [C5] 문제 4. `Pipeline` 통째로 `GridSearchCV`
# ⌨️ 문제 4 — 전처리 옵션과 모델 옵션을 한 격자에서
from sklearn.model_selection import GridSearchCV

# 여기에 코드를 작성하세요 (격자 정의 → GridSearchCV → 결과표 → best)
# 1)
feat = shoppers.copy()
feat["has_page_value"] = (feat["PageValues"] > 0).astype(int)

num_cols_ext = NUM_COLS + ["has_page_value"]

pre = ColumnTransformer([
    ("num", "passthrough", num_cols_ext),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT_COLS),
])

pipe = Pipeline([
    ("pre", pre),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)),
])

# 2) GridSearchCV
param_grid = {
    "pre__cat__min_frequency": [10, 50],
    "clf__min_samples_leaf": [5, 20, 50],
}

gs = GridSearchCV(
    pipe, param_grid,
    scoring="average_precision",
    cv=5,
    n_jobs=-1,
    refit=True,
)
gs.fit(feat[num_cols_ext + CAT_COLS], y)

# 3) 결과표
results = pd.DataFrame(gs.cv_results_)[
    ["param_pre__cat__min_frequency", "param_clf__min_samples_leaf",
     "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False)
print(results)

print("\nbest_params_:", gs.best_params_)
print("best_score_:", gs.best_score_)

# 4) 선택된 모델의 F1도 같은 5겹에서 측정
from sklearn.model_selection import cross_val_score
best_pipe = gs.best_estimator_
f1_scores = cross_val_score(best_pipe, feat[num_cols_ext + CAT_COLS], y, cv=5, scoring="f1")
print("선택 모델 F1 (내부 CV, 참고용):", f1_scores.mean())

   param_pre__cat__min_frequency  param_clf__min_samples_leaf  \
1                             50                            5   
0                             10                            5   
3                             50                           20   
2                             10                           20   
4                             10                           50   
5                             50                           50   

   mean_test_score  std_test_score  
1         0.737451        0.087928  
0         0.737106        0.088529  
3         0.732714        0.088248  
2         0.731764        0.085184  
4         0.722944        0.083610  
5         0.722305        0.086789  

best_params_: {'clf__min_samples_leaf': 5, 'pre__cat__min_frequency': 50}
best_score_: 0.7374509050360163
선택 모델 F1 (내부 CV, 참고용): 0.6378125907577962


<details>
<summary>(클릭) 💡 힌트</summary>

- `Pipeline` 안의 손잡이는 **이중 밑줄**로 지정합니다: `단계이름__파라미터`. 중첩되면 계속 이어 붙입니다 — `pre__cat__min_frequency`는 "`pre` 단계의 `cat` 갈래의 `min_frequency`"입니다.
- 단계 이름은 만들 때 준 이름 그대로입니다(`"pre"`, `"cat"`, `"clf"`).
- 결과는 `gs.cv_results_`에 들어 있습니다. `pd.DataFrame(gs.cv_results_)`에서 `param_*`·`mean_test_score`·`std_test_score` 열만 뽑으면 깔끔합니다.
- `n_jobs=-1`을 주면 조합을 병렬로 돌려 훨씬 빠릅니다.

</details>

> 🎯 **[C5] 확인하기**  
> **이 탐색에서는 5가 선택됐습니다.**
>
> 선택값 변화의 원인을 점검합니다. 피처 구성과 탐색 공간이 달라지면 선택되는 하이퍼파라미터도 달라질 수 있습니다. 이번 결과만으로 선택값 변화의 원인을 피처 수 하나로 단정할 수는 없습니다.
>
> 여기서 챙길 원칙 하나. **교차 검증에서 선택되는 하이퍼파라미터는 피처 구성이 바뀌면 함께 바뀝니다.** 피처를 바꾼 뒤에는 이전 설정을 그대로 고정하지 않고 다시 검증합니다. 따라서 튜닝은 피처 검토 뒤에 수행합니다.
>
> 이번 탐색의 상위 두 조합에서 `min_frequency`에 따른 평균 차이는 0.0004입니다. **전처리 손잡이도 격자에 넣어 봤기 때문에** "영향이 작다"고 말할 수 있는 것이지, 이 차이의 안정성은 반복 검증이나 외부 검증으로 추가 확인해야 합니다.

# 문제 5. 실험 기록표 · 모델 저장 · 모델 카드 v5

오늘 여러 실험을 했습니다. 그 기록을 남기고, 최종 모델을 파일로 저장합니다. 기록과 학습된 `Pipeline`을 함께 남기면 실험 조건과 예측 절차를 재현하기 쉬워집니다.

```
[문제 5]
1) 지금까지 모은 log를 DataFrame으로 만들어 실험 기록표를 출력하고 CSV로 저장합니다.
   → experiment_log_v5.csv  (열: 조건 → 지표 → 결정)
2) GridSearchCV의 최적 Pipeline을 joblib으로 저장합니다.
   → purchase_pipeline.joblib   (compress 옵션을 지정합니다)
3) 저장한 파일을 다시 불러와 예측이 되는지 확인합니다.
4) 아래 모델 카드 v5 템플릿을 채웁니다.
```

> ⚠️ **주의하기 — 선택 점수와 최종 평가**  
> `best_score_`는 같은 데이터에서 후보를 비교해 얻은 내부 CV 점수이므로 낙관적으로 편향될 수 있습니다. 최종 일반화 성능은 별도 테스트셋이나 바깥쪽 교차 검증(Nested CV)으로 평가합니다.

> ⚠️ **주의하기 — 저장 파일을 다루는 규칙**  
> 저장한 `.joblib`은 파일을 여는 것만으로 그 안에 담긴 코드가 실행될 수 있습니다. **출처를 신뢰할 수 없는 파일은 열지 않습니다.** 또한 저장한 환경과 여는 환경의 `scikit-learn` 버전이 다르면 로드에 실패할 수 있으므로, 모델 파일을 남길 때 버전도 함께 기록합니다.

▶️ **코드 실행하기 · 코드 셀 6 [C6]**

In [11]:
# [C6] 문제 5. 실험 기록표 · 모델 저장 · 모델 카드 v5
# ⌨️ 문제 5 — 기록표 CSV + Pipeline joblib 저장·재로드
import joblib
import sklearn

# 여기에 코드를 작성하세요 (log → DataFrame → CSV / 최적 Pipeline 저장 → 재로드 확인)
# 1) log → DataFrame → CSV
log = [
    {"조건": "수치형만 (기준)", "F1": base_f1, "AP": base_ap,
     "결정": "기준선"},
    {"조건": "수치형+범주형 (문제1)", "F1": res_full["test_f1"].mean(),
     "AP": res_full["test_average_precision"].mean(),
     "결정": "채택 - 범주형 추가로 AP 상승"},
    {"조건": "가설A: has_page_value", "F1": f1_A, "AP": ap_A,
     "결정": "기각 - 개선폭 노이즈 수준(Δ+0.0019)"},
    {"조건": "가설B: total_pages", "F1": f1_B, "AP": ap_B,
     "결정": "기각 - 오히려 하락(Δ-0.0043)"},
    {"조건": "타깃인코딩 (누수 버전)", "F1": f1_leak, "AP": ap_leak,
     "결정": "기각 - 전체데이터로 계산해 검증 정답 유출, 점수 부풀려짐"},
    {"조건": "타깃인코딩 (교차적합)", "F1": f1_te, "AP": ap_te,
     "결정": "기각 - 진짜 개선폭 미미(ΔAP+0.0011)"},
    {"조건": "GridSearchCV 최적 (문제4)", "F1": f1_scores.mean(), "AP": gs.best_score_,
     "결정": "채택(잠정) - 내부 CV 최고점, 최종 성능 아님"},
]

log_df = pd.DataFrame(log)
print(log_df)
log_df.to_csv("experiment_log_v5.csv", index=False, encoding="utf-8-sig")

# 2) 최적 Pipeline 저장
joblib.dump(gs.best_estimator_, "purchase_pipeline.joblib", compress=3)
print("저장 시 sklearn 버전:", sklearn.__version__)

# 3) 재로드 확인
loaded_pipe = joblib.load("purchase_pipeline.joblib")

# 학습 때와 같은 열 구성으로 넣어야 함
sample = feat[num_cols_ext + CAT_COLS].iloc[:5]
preds = loaded_pipe.predict(sample)
probs = loaded_pipe.predict_proba(sample)[:, 1]

print("예측 클래스:", preds)
print("예측 확률(구매=1):", probs)

                      조건        F1        AP  \
0              수치형만 (기준)  0.612013  0.717083   
1          수치형+범주형 (문제1)  0.544683  0.716203   
2    가설A: has_page_value  0.613923  0.718309   
3       가설B: total_pages  0.607733  0.715734   
4          타깃인코딩 (누수 버전)  0.657156  0.754318   
5           타깃인코딩 (교차적합)  0.619293  0.718214   
6  GridSearchCV 최적 (문제4)  0.637813  0.737451   

                                  결정  
0                                기준선  
1                 채택 - 범주형 추가로 AP 상승  
2          기각 - 개선폭 노이즈 수준(Δ+0.0019)  
3              기각 - 오히려 하락(Δ-0.0043)  
4  기각 - 전체데이터로 계산해 검증 정답 유출, 점수 부풀려짐  
5         기각 - 진짜 개선폭 미미(ΔAP+0.0011)  
6       채택(잠정) - 내부 CV 최고점, 최종 성능 아님  
저장 시 sklearn 버전: 1.9.0
예측 클래스: [0 0 0 0 0]
예측 확률(구매=1): [0.00073054 0.00750803 0.00232713 0.00664716 0.01371698]


In [12]:
print(gs.best_params_)
print(gs.best_score_)

results_sorted = pd.DataFrame(gs.cv_results_).sort_values("mean_test_score", ascending=False)
print(results_sorted[["mean_test_score", "std_test_score"]].iloc[0])

print(f1_scores.mean(), f1_scores.std())

{'clf__min_samples_leaf': 5, 'pre__cat__min_frequency': 50}
0.7374509050360163
mean_test_score    0.737451
std_test_score     0.087928
Name: 1, dtype: float64
0.6378125907577962 0.03862555614169843


<details>
<summary>(클릭) 💡 힌트</summary>

- `pd.DataFrame(log)`로 바로 표가 됩니다. `결정` 열은 손으로 채워 넣습니다(채택/기각과 한 줄 이유).
- `joblib.dump(gs.best_estimator_, "purchase_pipeline.joblib", compress=3)` — 압축을 주지 않으면 20MB가 넘습니다.
- **`Pipeline` 통째로** 저장해야 합니다. 모델만 저장하면 전처리를 다시 만들어야 하고, 그때 설정이 어긋나면 그것도 사고입니다.
- 재로드는 `joblib.load(...)`이고, 예측에 넣는 데이터는 **학습할 때와 같은 열 구성**이어야 합니다.

</details>

**모델 카드 v5 템플릿** — 아래 빈칸을 채워 이 셀에 완성합니다 (셀을 더블클릭해 편집).

```markdown
## 모델 카드 v5 — 구매 전환 예측

- 데이터: UCI Online Shoppers (12,330세션, 원본 예측 피처 17개 검토)
- 검증 방식: StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
- 주 지표: AP(`average_precision`) · F1 (정확도는 참고)
- **전처리: ColumnTransformer — 수치 { 10 }개 passthrough / 범주 { 7 }개 One-Hot(min_frequency={ 20 })**
  - **변환 후 피처 { 66 }개**
- **파생변수(가설 → 결과)**
  - 가설 A `has_page_value`: { PageValues>0 여부가 추가 표현으로 유용할 것 } → Δf1 { +0.0019 } · ΔAP { +0.0012 } → **{ 기각 }**
  - 가설 B `total_pages`: { 총 탐색 페이지 수가 구매와 관련 있을 것 } → Δf1 { -0.0041 } · ΔAP { -0.0013 } → **{ 기각 }**
- **누수 점검: combo(Month×TrafficType×Region) 715개 조합 중 464개가 10건 미만. 전체 데이터로 조합별 구매율을 계산해 넣은  
              누수 버전은 기준 대비 ΔF1 +0.0451·ΔAP +0.0372로 부풀려짐 → 교차 적합 버전은 기준 대비 ΔF1 +0.0077·ΔAP  
              +0.0011로 개선폭 미미**
  - **학습되는 전처리의 `fit`이 `Pipeline` 안에서 일어나는가: { 예 }**
- **튜닝: GridSearchCV(scoring="average_precision"), 격자 { pre__cat__min_frequency: [10,50],  
          clf__min_samples_leaf: [5,20,50] } → best { pre__cat__min_frequency: 50, clf__min_samples_leaf: 5 }**
- 내부 CV 추정: AP { 0.737 } ± { 0.088 } / F1 { 0.638 } ± { 0.039 }
- 최종 평가: 아직 미실시 - 별도 테스트셋 또는 Nested CV로 추가 검증 필요
- **산출물: purchase_pipeline.joblib (Pipeline 통째, compress=3) · experiment_log_v5.csv**
- 한계 & 다음 단계: PageValues가 실제 예측 시점(구매 확정 전)에 이미 가용한 값인지 확인 필요(시점 누수 후보로 분류됨).  
                  향후 외부 데이터로 검증, 피처 중요도 기반 해석 추가 예정.
- AI 사용 내역   
  무엇을 물었나: Pipeline/ColumnTransformer 구성, 타깃 인코딩 누수 재현 방법, GridSearchCV 파라미터 문법에 대해 질의     
  fit 위치를 어떻게 검증했나: 전처리 fit이 Pipeline 안에서 폴드마다 새로 일어나는지 cross_validate에 Pipeline 전체를 넘기는 방식으로 검증
```

**제출:** 노트북 · `experiment_log_v5.csv` · `purchase_pipeline.joblib`을 개인 공개 저장소 main에 커밋·푸시하고, 저장소·커밋 링크를 제출합니다.

**스스로 점검하는 기준**

| 축        | 기준                                                                |
| --------- | ------------------------------------------------------------------- |
| 가설 우선 | 코드보다 가설을 먼저 적었는가                                       |
| 실험 격리 | 파생변수를 하나씩 따로 재서 원인을 가렸는가                         |
| 누수 규율 | 학습되는 전처리의 `fit`이 `Pipeline` 안에 있는가, 그것을 명시했는가 |
| 재현성    | 저장한 파일만으로 남이 같은 예측을 얻을 수 있는가                   |

> 🚀 **직접 확장하기**  
> 문제 1에서 범주형을 넣자 F1이 떨어졌습니다. 검증 데이터에서 **후보 임계값을 다시 선택**하고 별도 테스트셋에서 확인합니다. 그러면 "지표가 떨어졌다"와 "모델이 나빠졌다"가 다른 말이라는 것을 숫자로 확인할 수 있습니다.

오늘 여러분은 사용하지 않았던 범주형 피처를 연결했고, 가설 두 개 중 하나를 정직하게 기각했으며, **누수 버전의 개선 +0.011이 교차 적합 평가에서 재현되지 않음을 확인했습니다.**

성능 개선과 누수 감사는 같은 실험 기록 안에서 함께 수행해야 합니다. 그래야 점수의 계산 조건과 채택 근거를 추적할 수 있습니다.

오늘도 한 걸음, 수고하셨습니다! 🎉

---

<sub>© 2026 모두의연구소(MODULABS). All rights reserved.<br>
기획·제작: 교육퍼실리테이터팀 이진영 (jy.lee@modulabs.co.kr)<br>
본 자료는 생성형 AI를 활용해 제작되었고, 제작자의 검수를 거쳐 완성되었습니다.<br>
무단 복제 및 배포를 금합니다.</sub>